# Agentic AI-Based EV Dynamic Tariff Optimization

This notebook provides a complete walk-through of the data preprocessing, exploratory data analysis, machine learning demand forecasting, and simulation of the Agentic AI pricing feedback loop on both the **ACN-Data** and **UrbanEV** datasets.

## 1. Environment Setup & Dataset Cloning

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

# Auto-configure workspace for cloud environments
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_COLAB or IS_KAGGLE:
    print('Setting up cloud environment...')
    !pip install -q openpyxl lightgbm
    
    # Fetch only the datasets directory using a sparse checkout
    if not os.path.exists('Datasets OP_26 Analytics'):
        print('Fetching datasets from GitHub...');
        !git clone --filter=blob:none --no-checkout https://github.com/iaryan08/agentic-ev-tariff-optimizer.git temp_repo
        !cd temp_repo && git sparse-checkout set "Datasets OP_26 Analytics" && git checkout main
        !mv temp_repo/"Datasets OP_26 Analytics" ./
        !rm -rf temp_repo
        print('Datasets fetched successfully.')

def get_dataset_path(rel_path_from_root):
    if os.path.basename(os.getcwd()) == 'notebooks':
        return os.path.join('..', rel_path_from_root)
    return rel_path_from_root

sns.set_theme(style='darkgrid')
print('Environment initialized.')


## 2. Core Modules Definition (Embedded python files)

### 2.1 Data Preprocessing Module

### 2.1.1 ACN Data Preprocessing


In [ ]:
import pandas as pd
import numpy as np
import os

def preprocess_acn(file_path):
    print(f"Loading ACN dataset: {file_path}")
    df = pd.read_excel(file_path)
    
    # Clean redundant metadata columns
    empty_cols = ['_meta', 'end', 'min_kWh', 'start', '_items', 'userInputs']
    df = df.drop(columns=[col for col in empty_cols if col in df.columns], errors='ignore')
    
    # Forward-fill session information for nested user updates
    session_cols = ['_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 
                    'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID']
    df[session_cols] = df[session_cols].ffill()
    
    # Parse timestamps and compute durations
    for col in ['connectionTime', 'disconnectTime', 'doneChargingTime', 'modifiedAt', 'requestedDeparture']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            
    df['session_duration_hr'] = (df['disconnectTime'] - df['connectionTime']).dt.total_seconds() / 3600.0
    df['charging_duration_hr'] = (df['doneChargingTime'] - df['connectionTime']).dt.total_seconds() / 3600.0
    
    # Handle outlier/negative durations
    df['session_duration_hr'] = df['session_duration_hr'].clip(lower=0.1)
    df['charging_duration_hr'] = df['charging_duration_hr'].clip(lower=0.1)
    
    df = df.dropna(subset=['connectionTime', 'disconnectTime', 'kWhDelivered'])
    df_sessions = df.drop_duplicates(subset=['sessionID']).copy()
    print(f"ACN loaded: {len(df_sessions)} unique sessions.")
    
    # Vectorized hourly aggregation
    min_time = df_sessions['connectionTime'].min().floor('h')
    max_time = df_sessions['disconnectTime'].max().ceil('h')
    hourly_range = pd.date_range(start=min_time, end=max_time, freq='h')
    
    print(f"Aggregating ACN sessions into hourly intervals ({len(hourly_range)} hours)...")
    df_sessions['charging_rate_kW'] = df_sessions['kWhDelivered'] / df_sessions['charging_duration_hr']
    
    conn_times = df_sessions['connectionTime'].values
    disc_times = df_sessions['disconnectTime'].values
    done_times = df_sessions['doneChargingTime'].values
    rates = df_sessions['charging_rate_kW'].values
    
    hourly_data = []
    for h in hourly_range:
        h_np = np.datetime64(h)
        h_next_np = np.datetime64(h + pd.Timedelta(hours=1))
        
        is_occupied = (conn_times < h_next_np) & (disc_times > h_np)
        is_charging = (conn_times < h_next_np) & (done_times > h_np)
        
        occupancy = np.sum(is_occupied)
        charging_load = np.sum(rates[is_charging])
        
        hourly_data.append({
            'timestamp': h,
            'occupancy': occupancy,
            'charging_load_kW': charging_load,
            'energy_delivered_kWh': charging_load
        })
        
    df_hourly = pd.DataFrame(hourly_data)
    
    df_hourly['hour'] = df_hourly['timestamp'].dt.hour
    df_hourly['day'] = df_hourly['timestamp'].dt.day
    df_hourly['month'] = df_hourly['timestamp'].dt.month
    df_hourly['dayofweek'] = df_hourly['timestamp'].dt.dayofweek
    df_hourly['is_weekend'] = df_hourly['dayofweek'].isin([5, 6]).astype(int)
    
    print(f"ACN aggregation complete. Shape: {df_hourly.shape}")
    return df_sessions, df_hourly


### 2.1.2 UrbanEV Data Preprocessing


In [ ]:
def preprocess_urbanev(data_dir):
    print(f"Loading UrbanEV dataset from {data_dir}...")
    
    occupancy = pd.read_csv(os.path.join(data_dir, "occupancy.csv"))
    volume = pd.read_csv(os.path.join(data_dir, "volume.csv"))
    price = pd.read_csv(os.path.join(data_dir, "price.csv"))
    duration = pd.read_csv(os.path.join(data_dir, "duration.csv"))
    time_df = pd.read_csv(os.path.join(data_dir, "time.csv"))
    info = pd.read_csv(os.path.join(data_dir, "information.csv"))
    
    # Scale price matrix to INR baseline (CNY to INR 15x scaling maps to average ~₹14.38/kWh)
    price_cols = [c for c in price.columns if c != 'timestamp']
    price[price_cols] = price[price_cols] * 15.0
    
    capacity_dict = dict(zip(info['grid'].astype(str), info['count']))
    fast_dict = dict(zip(info['grid'].astype(str), info['fast_count']))
    slow_dict = dict(zip(info['grid'].astype(str), info['slow_count']))
    cbd_dict = dict(zip(info['grid'].astype(str), info['CBD']))
    area_dict = dict(zip(info['grid'].astype(str), info['area']))
    dynamic_pricing_dict = dict(zip(info['grid'].astype(str), info['dynamic_pricing']))
    
    timestamps = pd.to_datetime(time_df[['year', 'month', 'day', 'hour', 'minute', 'second']])
    grid_ids = [c for c in occupancy.columns if c != 'timestamp']
    
    print("Reshaping UrbanEV matrices to long-format time-series...")
    records = []
    for grid in grid_ids:
        grid_str = str(grid)
        cap = capacity_dict.get(grid_str, 1.0)
        fast = fast_dict.get(grid_str, 0)
        slow = slow_dict.get(grid_str, 0)
        cbd = cbd_dict.get(grid_str, 0)
        area = area_dict.get(grid_str, 1.0)
        dyn_p = dynamic_pricing_dict.get(grid_str, 0)
        
        grid_df = pd.DataFrame({
            'timestamp': timestamps,
            'grid_id': grid_str,
            'occupancy': occupancy[grid],
            'volume': volume[grid],
            'price_INR': price[grid],
            'duration_min': duration[grid]
        })
        
        grid_df['capacity'] = cap
        grid_df['fast_count'] = fast
        grid_df['slow_count'] = slow
        grid_df['CBD'] = cbd
        grid_df['area'] = area
        grid_df['dynamic_pricing_enabled'] = dyn_p
        
        # Derived features
        grid_df['utilization_rate'] = grid_df['occupancy'] / grid_df['capacity']
        grid_df['occupancy_density'] = grid_df['occupancy'] / area
        
        records.append(grid_df)
        
    df_long = pd.concat(records, ignore_index=True)
    
    df_long['hour'] = df_long['timestamp'].dt.hour
    df_long['day'] = df_long['timestamp'].dt.day
    df_long['month'] = df_long['timestamp'].dt.month
    df_long['dayofweek'] = df_long['timestamp'].dt.dayofweek
    df_long['is_weekend'] = df_long['dayofweek'].isin([5, 6]).astype(int)
    df_long['queue_length_proxy'] = (df_long['occupancy'] - df_long['capacity']).clip(lower=0.0)
    
    print(f"UrbanEV long-format complete. Shape: {df_long.shape}")
    
    return df_long, {
        'occupancy': occupancy,
        'volume': volume,
        'price': price,
        'duration': duration,
        'time': time_df,
        'info': info,
        'timestamps': timestamps
    }


### 2.2 Demand Prediction Agent Module

### 2.2.1 Class Initialization & Feature Engineering


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import os

class DemandPredictionAgent:
    def __init__(self, target_type='urbanev'):
        self.target_type = target_type
        self.model_occupancy = None
        self.model_volume = None
        self.label_encoders = {}
        
    def _create_features(self, df):
        sort_cols = ['grid_id', 'timestamp'] if 'grid_id' in df.columns else ['timestamp']
        df = df.copy().sort_values(by=sort_cols)
        
        group_col = 'grid_id' if 'grid_id' in df.columns else None
        occ_col = 'occupancy'
        vol_col = 'volume' if 'volume' in df.columns else 'charging_load_kW'
        
        # Lag features and rolling means
        if group_col:
            df['occ_lag_1'] = df.groupby(group_col)[occ_col].shift(1)
            df['occ_lag_2'] = df.groupby(group_col)[occ_col].shift(2)
            df['occ_lag_24'] = df.groupby(group_col)[occ_col].shift(24)
            
            df['vol_lag_1'] = df.groupby(group_col)[vol_col].shift(1)
            df['vol_lag_2'] = df.groupby(group_col)[vol_col].shift(2)
            df['vol_lag_24'] = df.groupby(group_col)[vol_col].shift(24)
            
            df['occ_roll_mean_3'] = df.groupby(group_col)['occ_lag_1'].transform(lambda x: x.rolling(3).mean())
            df['vol_roll_mean_3'] = df.groupby(group_col)['vol_lag_1'].transform(lambda x: x.rolling(3).mean())
        else:
            df['occ_lag_1'] = df[occ_col].shift(1)
            df['occ_lag_2'] = df[occ_col].shift(2)
            df['occ_lag_24'] = df[occ_col].shift(24)
            
            df['vol_lag_1'] = df[vol_col].shift(1)
            df['vol_lag_2'] = df[vol_col].shift(2)
            df['vol_lag_24'] = df[vol_col].shift(24)
            
            df['occ_roll_mean_3'] = df['occ_lag_1'].rolling(3).mean()
            df['vol_roll_mean_3'] = df['vol_lag_1'].rolling(3).mean()
            
        df = df.ffill().bfill()
        return df


### 2.2.2 Model Training & Evaluation


In [ ]:
def train(self, df):
    print(f"Training Demand Prediction models for {self.target_type}...")
    df_feat = self._create_features(df)
    
    feature_cols = ['hour', 'dayofweek', 'is_weekend', 'month', 
                    'occ_lag_1', 'occ_lag_2', 'occ_lag_24', 
                    'vol_lag_1', 'vol_lag_2', 'vol_lag_24',
                    'occ_roll_mean_3', 'vol_roll_mean_3']
    
    if self.target_type == 'urbanev':
        feature_cols += ['capacity', 'fast_count', 'slow_count', 'CBD', 'area', 'dynamic_pricing_enabled']
        if 'grid_id' in df_feat.columns:
            le = LabelEncoder()
            df_feat['grid_id_encoded'] = le.fit_transform(df_feat['grid_id'])
            self.label_encoders['grid_id'] = le
            feature_cols += ['grid_id_encoded']
            
    # Split 80% train / 20% test chronologically
    n_samples = len(df_feat)
    train_size = int(n_samples * 0.8)
    
    train_df = df_feat.iloc[:train_size]
    test_df = df_feat.iloc[train_size:]
    
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    
    occ_target = 'occupancy'
    vol_target = 'volume' if 'volume' in df.columns else 'charging_load_kW'
    
    y_train_occ = train_df[occ_target]
    y_test_occ = test_df[occ_target]
    y_train_vol = train_df[vol_target]
    y_test_vol = test_df[vol_target]
    
    self.model_occupancy = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.08, random_state=42, verbose=-1)
    self.model_volume = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.08, random_state=42, verbose=-1)
    
    self.model_occupancy.fit(X_train, y_train_occ)
    self.model_volume.fit(X_train, y_train_vol)
    
    preds_occ = self.model_occupancy.predict(X_test)
    preds_vol = self.model_volume.predict(X_test)
    
    metrics = {}
    for target_name, y_true, y_pred in [('occupancy', y_test_occ, preds_occ), ('volume', y_test_vol, preds_vol)]:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        metrics[target_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        print(f"[{target_name.upper()}] Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")
        
    return metrics

DemandPredictionAgent.train = train


### 2.2.3 Model Inference


In [ ]:
def predict(self, df):
    df_feat = self._create_features(df)
    feature_cols = ['hour', 'dayofweek', 'is_weekend', 'month', 
                    'occ_lag_1', 'occ_lag_2', 'occ_lag_24', 
                    'vol_lag_1', 'vol_lag_2', 'vol_lag_24',
                    'occ_roll_mean_3', 'vol_roll_mean_3']
    
    if self.target_type == 'urbanev':
        feature_cols += ['capacity', 'fast_count', 'slow_count', 'CBD', 'area', 'dynamic_pricing_enabled']
        if 'grid_id' in df_feat.columns:
            le = self.label_encoders['grid_id']
            df_feat['grid_id_encoded'] = df_feat['grid_id'].map(lambda s: le.transform([s])[0] if s in le.classes_ else 0)
            feature_cols += ['grid_id_encoded']
            
    df_feat['pred_occupancy'] = self.model_occupancy.predict(df_feat[feature_cols])
    df_feat['pred_volume'] = self.model_volume.predict(df_feat[feature_cols]).clip(min=0.0)
    
    if 'capacity' in df_feat.columns:
        df_feat['pred_utilization_rate'] = df_feat['pred_occupancy'] / df_feat['capacity']
    else:
        max_occ = df_feat['occupancy'].max() if df_feat['occupancy'].max() > 0 else 1.0
        df_feat['pred_utilization_rate'] = df_feat['pred_occupancy'] / max_occ
        
    # Congestion sigmoid center boundary is 80% utilization
    df_feat['congestion_probability'] = 1 / (1 + np.exp(-10 * (df_feat['pred_utilization_rate'] - 0.8)))
    return df_feat

DemandPredictionAgent.predict = predict


### 2.3 Tariff Pricing Agent Module

In [ ]:
import numpy as np

class TariffPricingAgent:
    def __init__(self, baseline_price=15.0, max_price=25.0, min_price=10.0, elasticity=0.25):
        self.baseline_price = baseline_price
        self.max_price = max_price
        self.min_price = min_price
        self.elasticity = elasticity
        
    def set_elasticity(self, elasticity):
        self.elasticity = elasticity
        
    def calculate_tariff(self, predicted_utilization):
        # Surge boundary: >80% utilization. Discounts boundary: <30% utilization.
        tariffs = np.where(
            predicted_utilization > 0.8,
            self.baseline_price + (self.max_price - self.baseline_price) * ((predicted_utilization - 0.8) / 0.2),
            np.where(
                predicted_utilization < 0.3,
                self.min_price + (self.baseline_price - self.min_price) * (predicted_utilization / 0.3),
                self.baseline_price
            )
        )
        return np.clip(tariffs, self.min_price, self.max_price)
        
    def simulate_demand_response(self, predicted_demand, dynamic_price):
        price_diff_ratio = (dynamic_price - self.baseline_price) / self.baseline_price
        elastic_demand = predicted_demand * (1.0 - self.elasticity * price_diff_ratio)
        return np.clip(elastic_demand, 0.0, None)


### 2.4 Monitoring & Learning Agent Module

### 2.4.1 Metrics Evaluation


In [ ]:
import numpy as np

class MonitoringLearningAgent:
    def __init__(self, target_type='urbanev'):
        self.target_type = target_type
        self.history = []
        
    def evaluate_episode(self, df_results):
        occ_col = 'occupancy'
        vol_col = 'volume' if 'volume' in df_results.columns else 'charging_load_kW'
        p_base = 15.0
        
        p_dyn = df_results['dynamic_price']
        vol_base = df_results[vol_col]
        vol_dyn = df_results['dynamic_volume']
        occ_base = df_results[occ_col]
        occ_dyn = df_results['dynamic_occupancy']
        cap = df_results['capacity'] if 'capacity' in df_results.columns else max(1.0, df_results['occupancy'].max())
        
        # 1. Financial Performance
        revenue_base = np.sum(vol_base * p_base)
        revenue_dyn = np.sum(vol_dyn * p_dyn)
        revenue_gain_pct = ((revenue_dyn - revenue_base) / max(1.0, revenue_base)) * 100.0
        
        # 2. Utilization Metrics
        util_base = np.mean(occ_base / cap)
        util_dyn = np.mean(occ_dyn / cap)
        
        # 3. Off-Peak Shift (base utilization < 30%)
        off_peak_mask = (occ_base / cap) < 0.3
        vol_base_offpeak = np.sum(vol_base[off_peak_mask])
        vol_dyn_offpeak = np.sum(vol_dyn[off_peak_mask])
        off_peak_uplift_pct = (((vol_dyn_offpeak - vol_base_offpeak) / vol_base_offpeak) * 100.0) if vol_base_offpeak > 0 else 0.0
            
        # 4. Queue / Congestion Reduction (occupancy exceeding capacity)
        queue_base = np.maximum(0.0, occ_base - cap)
        queue_dyn = np.maximum(0.0, occ_dyn - cap)
        avg_queue_base = np.mean(queue_base)
        avg_queue_dyn = np.mean(queue_dyn)
        queue_reduction_pct = (((avg_queue_base - avg_queue_dyn) / avg_queue_base) * 100.0) if avg_queue_base > 0 else 0.0
            
        # 5. Observed Price Elasticity Proxy
        pct_vol_change = (vol_dyn - vol_base) / np.maximum(1.0, vol_base)
        pct_price_change = (p_dyn - p_base) / p_base
        active_pricing = np.abs(pct_price_change) > 0.01
        observed_elasticity = -np.mean(pct_vol_change[active_pricing] / pct_price_change[active_pricing]) if np.any(active_pricing) else 0.0
            
        # 6. Pricing Efficiency (Rev/kWh)
        pricing_efficiency_base = revenue_base / max(1.0, np.sum(vol_base))
        pricing_efficiency_dyn = revenue_dyn / max(1.0, np.sum(vol_dyn))
        
        metrics = {
            'Revenue_Baseline': revenue_base,
            'Revenue_Dynamic': revenue_dyn,
            'Revenue_Gain_Pct': revenue_gain_pct,
            'Utilization_Baseline': util_base,
            'Utilization_Dynamic': util_dyn,
            'Off_Peak_Uplift_Pct': off_peak_uplift_pct,
            'Queue_Base': avg_queue_base,
            'Queue_Dynamic': avg_queue_dyn,
            'Queue_Reduction_Pct': queue_reduction_pct,
            'Customer_Response_Rate': observed_elasticity,
            'Pricing_Efficiency_Baseline': pricing_efficiency_base,
            'Pricing_Efficiency_Dynamic': pricing_efficiency_dyn
        }
        
        self.history.append(metrics)
        return metrics


### 2.4.2 Feedback Loop & Parameter Tuning


In [ ]:
def update_parameters(self, current_metrics, pricing_agent):
    print("\n--- Monitoring & Learning Agent Parameter Tuning ---")
    new_elasticity = pricing_agent.elasticity
    
    # Tune off-peak response
    if current_metrics['Off_Peak_Uplift_Pct'] < 5.0 and current_metrics['Utilization_Baseline'] < 0.2:
        print("[Feedback] Low off-peak response. Increasing elasticity target.")
        new_elasticity += 0.05
        
    # Tune peak congestion response
    if current_metrics['Queue_Dynamic'] > 0.5:
        print("[Feedback] Persistent peak congestion. Boosting elasticity target.")
        new_elasticity += 0.05
        
    # Prevent demand collapse risk
    if current_metrics['Revenue_Gain_Pct'] < -10.0:
        print("[Feedback] High revenue degradation. Lowering pricing sensitivity to soften spikes.")
        new_elasticity -= 0.10
        
    new_elasticity = max(0.05, min(0.6, new_elasticity))
    pricing_agent.set_elasticity(new_elasticity)
    return new_elasticity

MonitoringLearningAgent.update_parameters = update_parameters


### 2.5 Simulation Pipeline Module


In [ ]:
def run_simulation(df_hourly, target_type='urbanev'):
    print(f'\n=== Running Simulation Pipeline for {target_type.upper()} ===')
    pred_agent = DemandPredictionAgent(target_type=target_type)
    pricing_agent = TariffPricingAgent()
    monitoring_agent = MonitoringLearningAgent(target_type=target_type)
    
    pred_agent.train(df_hourly)
    df_pred = pred_agent.predict(df_hourly)
    
    train_size = int(len(df_pred) * 0.8)
    df_test = df_pred.iloc[train_size:].copy()
    
    df_test['dynamic_price'] = pricing_agent.calculate_tariff(df_test['pred_utilization_rate'])
    
    vol_col = 'volume' if 'volume' in df_test.columns else 'charging_load_kW'
    occ_col = 'occupancy'
    df_test['dynamic_volume'] = pricing_agent.simulate_demand_response(df_test[vol_col], df_test['dynamic_price'])
    
    vol_ratio = df_test['dynamic_volume'] / np.maximum(0.1, df_test[vol_col])
    df_test['dynamic_occupancy'] = (df_test[occ_col] * vol_ratio).clip(lower=0.0)
    
    episode_metrics = monitoring_agent.evaluate_episode(df_test)
    return df_test, episode_metrics


## 3. Data Processing & Exploratory Data Analysis

In [ ]:
acn_path = get_dataset_path('Datasets OP_26 Analytics/ACN Data_ 25 April 2018 to 16 Dec 2018/acndata_sessions.json.xlsx')
urbanev_path = get_dataset_path('Datasets OP_26 Analytics/UrbanEV_ SZ_districts')

os.makedirs('outputs', exist_ok=True)

# Process ACN Dataset
if os.path.exists(acn_path):
    df_sessions, df_hourly = preprocess_acn(acn_path)
    df_sessions.to_csv('outputs/acn_cleaned_sessions.csv', index=False)
    print('Saved outputs/acn_cleaned_sessions.csv')
else:
    df_sessions, df_hourly = None, None
    print(f'Warning: ACN dataset not found at {acn_path}')

# Process UrbanEV Dataset
if os.path.exists(urbanev_path):
    df_long, dict_matrices = preprocess_urbanev(urbanev_path)
else:
    df_long, dict_matrices = None, None
    print(f'Warning: UrbanEV dataset not found at {urbanev_path}')


In [ ]:
if df_sessions is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(df_sessions['kWhDelivered'], bins=30, ax=axes[0], color='forestgreen', kde=True)
    axes[0].set_title('ACN Energy Delivered Distribution')
    axes[0].set_xlabel('Energy (kWh)')
    
    sns.histplot(df_sessions['session_duration_hr'], bins=30, ax=axes[1], color='dodgerblue', kde=True)
    axes[1].set_title('ACN Session Duration Distribution')
    axes[1].set_xlabel('Duration (Hours)')
    
    plt.tight_layout()
    plt.show()


In [ ]:
if df_long is not None:
    hourly_profile = df_long.groupby('hour')[['occupancy', 'volume', 'price_INR']].mean().reset_index()
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    ax1.set_xlabel('Hour of Day')
    ax1.set_ylabel('Average Occupancy (Vehicles)', color='tab:blue')
    ax1.plot(hourly_profile['hour'], hourly_profile['occupancy'], color='tab:blue', linewidth=2)
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    
    ax2 = ax1.twinx()
    ax2.set_ylabel('Average Price (INR/kWh)', color='tab:green')
    ax2.plot(hourly_profile['hour'], hourly_profile['price_INR'], color='tab:green', linewidth=2, linestyle='--')
    ax2.tick_params(axis='y', labelcolor='tab:green')
    
    plt.title('UrbanEV Daily Occupancy and Price Profile')
    fig.tight_layout()
    plt.savefig('outputs/eda_profile.png', dpi=300, bbox_inches='tight')
    plt.show()


## 4. Agent Training & Evaluation

In [ ]:
if df_hourly is not None:
    print('Training Demand Prediction Model (ACN)...')
    pred_acn = DemandPredictionAgent(target_type='acn')
    acn_metrics = pred_acn.train(df_hourly)
    print('Evaluation metrics:', acn_metrics)
    
    # Run simulation to generate ACN deliverables
    df_test_acn, metrics_acn = run_simulation(df_hourly, target_type='acn')
    df_test_acn.to_csv('outputs/acn_simulation_results.csv', index=False)
    pd.DataFrame([metrics_acn]).to_csv('outputs/acn_metrics.csv', index=False)
    print('Saved ACN simulation CSV deliverables to outputs/')


In [ ]:
def aggregate_urbanev_to_hourly(df):
    df_copy = df.copy()
    df_copy['timestamp'] = df_copy['timestamp'].dt.floor('h')
    agg_rules = {
        'occupancy': 'mean',
        'volume': 'sum',
        'price_INR': 'mean',
        'capacity': 'first',
        'fast_count': 'first',
        'slow_count': 'first',
        'CBD': 'first',
        'area': 'first',
        'dynamic_pricing_enabled': 'first'
    }
    df_h = df_copy.groupby(['grid_id', 'timestamp']).agg(agg_rules).reset_index()
    
    df_h['hour'] = df_h['timestamp'].dt.hour
    df_h['day'] = df_h['timestamp'].dt.day
    df_h['month'] = df_h['timestamp'].dt.month
    df_h['dayofweek'] = df_h['timestamp'].dt.dayofweek
    df_h['is_weekend'] = df_h['dayofweek'].isin([5, 6]).astype(int)
    return df_h

if df_long is not None:
    print('Training Demand Prediction Model (UrbanEV)...')
    df_hourly_uev = aggregate_urbanev_to_hourly(df_long)
    pred_uev = DemandPredictionAgent(target_type='urbanev')
    uev_metrics = pred_uev.train(df_hourly_uev)
    print('Evaluation metrics:', uev_metrics)


## 5. Closed-Loop Simulation Pipeline

In [ ]:
def run_simulation(df_hourly, target_type='urbanev'):
    print(f'\n=== Running Simulation Pipeline for {target_type.upper()} ===')
    pred_agent = DemandPredictionAgent(target_type=target_type)
    pricing_agent = TariffPricingAgent()
    monitoring_agent = MonitoringLearningAgent(target_type=target_type)
    
    pred_agent.train(df_hourly)
    df_pred = pred_agent.predict(df_hourly)
    
    train_size = int(len(df_pred) * 0.8)
    df_test = df_pred.iloc[train_size:].copy()
    
    df_test['dynamic_price'] = pricing_agent.calculate_tariff(df_test['pred_utilization_rate'])
    
    vol_col = 'volume' if 'volume' in df_test.columns else 'charging_load_kW'
    occ_col = 'occupancy'
    df_test['dynamic_volume'] = pricing_agent.simulate_demand_response(df_test[vol_col], df_test['dynamic_price'])
    
    vol_ratio = df_test['dynamic_volume'] / np.maximum(0.1, df_test[vol_col])
    df_test['dynamic_occupancy'] = (df_test[occ_col] * vol_ratio).clip(lower=0.0)
    
    episode_metrics = monitoring_agent.evaluate_episode(df_test)
    return df_test, episode_metrics

if df_long is not None:
    df_test_uev, metrics_uev = run_simulation(df_hourly_uev, target_type='urbanev')
    print('\nMetrics:')
    for k, v in metrics_uev.items():
        print(f'{k}: {v}')
        
    # Save UrbanEV deliverables
    df_test_uev.to_csv('outputs/urbanev_simulation_results.csv', index=False)
    pd.DataFrame([metrics_uev]).to_csv('outputs/urbanev_metrics.csv', index=False)
    print('Saved UrbanEV simulation CSV deliverables to outputs/')


In [ ]:
if df_long is not None:
    test_h = df_test_uev.groupby('hour')[['occupancy', 'dynamic_occupancy', 'dynamic_price']].mean().reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(test_h['hour'], test_h['occupancy'], color='orange', label='Fixed Price Baseline')
    axes[0].plot(test_h['hour'], test_h['dynamic_occupancy'], color='green', label='Dynamic Tariff')
    axes[0].set_xlabel('Hour of Day')
    axes[0].set_ylabel('Average Occupancy (Vehicles)')
    axes[0].set_title('Occupancy Profile Comparison')
    axes[0].legend()
    
    axes[1].plot(test_h['hour'], test_h['dynamic_price'], color='purple', label='Dynamic Price')
    axes[1].axhline(15, color='red', linestyle='--', label='Fixed Price Baseline')
    axes[1].set_xlabel('Hour of Day')
    axes[1].set_ylabel('Price (INR/kWh)')
    axes[1].set_title('Dynamic Tariff Schedule')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('outputs/simulation_results.png', dpi=300, bbox_inches='tight')
    plt.show()
